## Intro to PySpark

A tour of the pieces you will use in the labs: reading data into a Spark DataFrame,
inspecting a query in the Spark UI, the low-level RDD API, the DataFrame API, and managed
tables. Run the cells in order.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [ ]:
import os
import subprocess
import sys

# A kernel started from an IDE or a GUI app never sources your shell profile, so JAVA_HOME
# can be missing here even though `java -version` works fine in a terminal. On macOS we ask
# Homebrew directly; on Windows/Linux you set it yourself (see getting_started/).
if "JAVA_HOME" not in os.environ:
    if sys.platform == "darwin":
        os.environ["JAVA_HOME"] = subprocess.check_output(
            ["brew", "--prefix", "openjdk@17"], text=True
        ).strip()
    else:
        raise RuntimeError("Set JAVA_HOME for your user account, then restart the kernel.")

os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ.get("PATH", "")
# Spark launches its Python workers by name; point them at this kernel's interpreter so it
# never falls back to a `python3` that does not exist (Windows) or is too old (macOS).
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("pyspark-demo")
    .master("local[*]")
    # A persistent catalog (a local Derby metastore under metastore_db/). Without it Spark
    # forgets its tables when the kernel dies but leaves their files in spark-warehouse/, so
    # re-running the saveAsTable cells below would fail with LOCATION_ALREADY_EXISTS.
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

In [ ]:
%%bash
uv run python get_data.py --dataset reviews

In [ ]:
# replace file path with your own if you have the dataset in a different location
df = spark.read.csv("data/y_amazon-google-large.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)

In [ ]:
df.describe().show()

In [ ]:
from pyspark.sql import functions as f

spark.sparkContext.setJobDescription("min/max ds aggregation")
df.agg(f.min("ds").alias("min_ds"), f.max("ds").alias("max_ds")).show()

### Reading the Spark UI

Spark UI: <http://localhost:4040> (or 4041, 4042... if you have more than one session running).

The **SQL / DataFrame** tab renders one query as a DAG of physical operators, each annotated
with live runtime metrics. This is Spark's most useful debugging view, because it shows
actual data volumes and timings rather than just the plan.

Reading the DAG for the min/max aggregation above, bottom-to-top (source → result):

| Node | Operator | What happened |
|---|---|---|
| `Scan csv` | reads the file | every row of the CSV is scanned; the metrics show rows read and bytes read |
| `HashAggregate` (partial) | per-partition min/max | each task reduces its own chunk locally, producing one row per partition |
| `Exchange` | shuffle | those few partial rows move to a single partition so they can be combined — only bytes, not megabytes |
| `HashAggregate` (final) | combines partials | merges the partial rows into the true global min/max |
| `AdaptiveSparkPlan` | root | wraps the whole thing; reports total query duration |

**Why two HashAggregates?**

This is Spark's standard partial → shuffle → final aggregation pattern. Instead of shuffling
every row to one place to compute min/max, each partition reduces its own chunk first (cheap
and fully parallel), and only the tiny partial results get shuffled and merged. That is why
the `Exchange` step moves bytes rather than megabytes.

**"Initial Plan" vs "Final Plan" in the text plan**

Spark 4.x has Adaptive Query Execution on by default: it plans optimistically, then re-optimises
using real statistics once a shuffle (`Exchange`) has actually run. That is why the plan text
shows both an `== Initial Plan ==` and a `== Final Plan ==`. For a query this small they end up
the same shape — there was not much to adapt.

**Why one query shows up as two jobs**

AQE submits each query stage (the plan segments separated by an `Exchange`) as its own Spark
job when it materialises, rather than one job for the whole query. So a single SQL execution
can appear as two job IDs at the bottom of the page.


In [ ]:
%%bash
uv run python get_data.py --dataset mnm

In [ ]:
mnm = spark.read.csv("data/mnm_dataset.csv", header=True, inferSchema=True)
# mnm.count() # 99999

color_agg_mnm = mnm.groupBy("Color").count().orderBy("count", ascending=False)
color_agg_mnm.show()

In [ ]:
color_agg_mnm_pdf = color_agg_mnm.toPandas()
color_agg_mnm_pdf.plot.bar(x="Color", y="count", legend=False)

In [ ]:
sns.set_theme(style="whitegrid")
sns.barplot(x="Color", y="count", data=color_agg_mnm_pdf)
plt.title("M&M Color Distribution")


## RDD - Low level API

In [ ]:
# Extract the SparkContext from the session
sc = spark.sparkContext

# Create an RDD of tuples (name, age)
data_rdd = sc.parallelize([("Brooke", 20), ("Denny", 31), ("Jules", 30), ("TD", 35), ("Brooke", 25)])
# Use map and reduceByKey transformations with their lambda
# expressions to aggregate and then compute average
ages_rdd = (
    data_rdd.map(lambda x: (x[0], (x[1], 1)))
    .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
    .map(lambda x: (x[0], x[1][0] / x[1][1]))
)

In [ ]:
ages_rdd.take(5)

## Dataframe API

In [ ]:
from pyspark.sql.functions import avg

# Create a DataFrame
data_df = spark.createDataFrame(
    [("Brooke", 20), ("Denny", 31), ("Jules", 30), ("TD", 35), ("Brooke", 25)], ["name", "age"]
)
# Group the same names together, aggregate their ages, and compute an average
avg_df = data_df.groupBy("name").agg(avg("age"))
# Show the results of the final execution
avg_df.show()

In [ ]:
parquet_table = "avg_age_by_name"
# mode("overwrite") so this cell can be re-run; without it a second run fails on an
# existing table.
avg_df.write.format("parquet").mode("overwrite").saveAsTable(parquet_table)

In [ ]:
from pyspark.sql import functions as f

dx = spark.read.format("parquet").table(parquet_table).withColumn("rnd", f.rand()).orderBy("rnd")
parquet_table2 = "avg_age_by_name2"
dx.write.format("parquet").mode("overwrite").saveAsTable(parquet_table2)

## Managed tables and databases

`USE learn_spark_db` switches the *current* database for the rest of the session, so the
tables created below live in `learn_spark_db` while `avg_age_by_name` above stayed in
`default`. Both are managed tables: Spark owns the data files under `spark-warehouse/` in
the repo root, and dropping the table deletes them.

The session above enabled Hive support, so the catalog itself is stored in `metastore_db/`
and survives a kernel restart — which is why these cells can be re-run. Delete both
`spark-warehouse/` and `metastore_db/` together if you ever want a clean slate; deleting
only one leaves the two out of sync.


In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS learn_spark_db")
spark.sql("USE learn_spark_db")

In [ ]:
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS managed_us_delay_flights_tbl (
        date STRING, delay INT, distance INT, origin STRING, destination STRING
    ) USING PARQUET
    """
)

In [ ]:
%%bash
uv run python get_data.py --dataset flights

In [ ]:
# Schema as defined in the preceding example
csv_file = "data/departuredelays.csv"
schema = "date STRING, delay INT, distance INT, origin STRING, destination STRING"
flights_df = spark.read.csv(csv_file, schema=schema)
# "overwrite" rather than "append": appending would duplicate every row on a re-run.
flights_df.write.saveAsTable("managed_us_delay_flights_tbl", format="parquet", mode="overwrite")

In [ ]:
df = spark.read.format("parquet").table("managed_us_delay_flights_tbl")
df.count()